In [16]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor, ConformalMetalogNewsvendor


ImportError: cannot import name 'ConformalMetalogNewsvendor' from 'tinyconformal.series' (/home/heylucasleao/tinyconformal/tinyconformal/series/__init__.py)

In [4]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# RandomForestQuantileRegressor

In [5]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [7]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     interval_cols=[
         ("RF-lo-90", "RF-hi-90"),
     ],
     n_windows=7,
     alpha=0.10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,interval_cols,"[('RF-lo-90', ...)]"
,n_windows,7
,alpha,0.1
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [8]:
cqr.predict_interval(h=7, X_df=test)

,unique_id,ds,RF-lo-90,RF-hi-90,RF-50,RF-lo-90-cqr,RF-hi-90-cqr
0,1,1960-01-01,337.00,436.5,396.0,331.30,442.20
1,1,1960-02-01,305.00,548.0,405.0,303.25,549.75
2,1,1960-03-01,301.00,559.0,396.0,259.00,601.00
3,1,1960-04-01,301.00,559.0,405.0,260.00,600.00
4,1,1960-05-01,300.15,559.0,379.5,265.15,594.00
5,1,1960-06-01,277.00,559.0,361.0,250.85,585.15
6,1,1960-07-01,276.10,559.0,363.0,205.10,630.00


,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RF,90%,0.1,0.833,282.354,465.688
1,RF-cqr,90%,0.1,1.000,317.288,317.288
